# ARGOS Access Control Testing

This notebook tests ARGOS query refinement using:
- `src/argos_abox_operator.py`
- `experiment/agentar_scale_sql/ScaleSQL/modules/argos_access_control.py`

Flow:
1. Setup ontology + DB ABOX context (from existing RDF or from `schema.json` + `access_control.json`).
2. Run SQL through ARGOS.
3. Inspect refined query + detected violations.

In [2]:
from __future__ import annotations

import json
import sys
from pathlib import Path
from pprint import pprint


def resolve_repo_root(start: Path | None = None) -> Path:
    base = (start or Path.cwd()).resolve()
    for candidate in [base, *base.parents]:
        if (candidate / "src" / "argos_abox_operator.py").exists():
            return candidate
    raise FileNotFoundError("Could not resolve repo root containing src/argos_abox_operator.py")


ROOT = resolve_repo_root()
SCALESQL_ROOT = ROOT / "experiment" / "agentar_scale_sql"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(SCALESQL_ROOT) not in sys.path:
    sys.path.insert(0, str(SCALESQL_ROOT))

print(f"ROOT={ROOT}")
print(f"SCALESQL_ROOT={SCALESQL_ROOT}")

ROOT=/home/nipdep/Dev/ARGOS
SCALESQL_ROOT=/home/nipdep/Dev/ARGOS/experiment/agentar_scale_sql


In [3]:
# ---- Configuration ----
DB_ID = "financial"
ROLE = "public"
SQL_QUERY = """
SELECT loan_id, amount
FROM loan
WHERE account_id IN (
  SELECT account_id FROM account WHERE district_id = 'd001'
)
""".strip()

ONTOLOGY_PATH = ROOT / "data/ontology_file/argos_v2.0.rdf"
BENCHMARK_ROOT = ROOT / "data/P3T2Q_benchmark/v0"

# If True, operator will parse an existing DB ABOX RDF directly.
# If False, operator will build ABOX from schema.json + access_control.json.
USE_EXISTING_DB_ABOX = False
EXISTING_DB_ABOX_PATH = ROOT / "experiment/outputs" / f"{DB_ID}_abox.rdf"

# Optional: save ABOX when building from benchmark configs.
SAVE_DB_ABOX = True
GENERATED_DB_ABOX_PATH = ROOT / "experiment/outputs" / "argos_notebook" / f"{DB_ID}_abox.rdf"

print(f"DB_ID={DB_ID} | ROLE={ROLE}")
print(f"ONTOLOGY_PATH={ONTOLOGY_PATH}")

DB_ID=financial | ROLE=public
ONTOLOGY_PATH=/home/nipdep/Dev/ARGOS/data/ontology_file/argos_v2.0.rdf


In [4]:
from rdflib import URIRef

from src.argos_abox_operator import ArgosABoxOperator, APUF
from scalesql.modules.argos_access_control import (
    ArgosAccessController,
    run_argos_access_control_case,
)


def infer_rdf_format(path: Path) -> str:
    suffix = path.suffix.lower()
    if suffix in {".ttl", ".turtle"}:
        return "turtle"
    if suffix == ".nt":
        return "nt"
    if suffix == ".n3":
        return "n3"
    return "xml"


def _detect_db_id_from_graph(operator: ArgosABoxOperator) -> str | None:
    for _, _, db_name in operator.graph.triples((None, APUF.DatabaseName, None)):
        value = str(db_name).strip()
        if value:
            return value
    return None


def _build_role_map_from_graph(operator: ArgosABoxOperator) -> dict[str, str]:
    role_map: dict[str, str] = {}
    for agent_uri, _, role_name in operator.graph.triples((None, APUF.AgentID, None)):
        role = str(role_name).strip().lower()
        if not role:
            continue
        agent_id = str(agent_uri).rsplit("#", 1)[-1]
        role_map[role] = agent_id
    return role_map


def setup_argos_operator(
    *,
    ontology_path: Path,
    benchmark_root: Path,
    db_id: str,
    use_existing_db_abox: bool,
    existing_db_abox_path: Path,
    save_db_abox: bool,
    generated_db_abox_path: Path,
) -> ArgosABoxOperator:
    operator = ArgosABoxOperator(ontology_path=ontology_path)

    if use_existing_db_abox:
        if not existing_db_abox_path.exists():
            raise FileNotFoundError(f"DB ABOX not found: {existing_db_abox_path}")
        operator.graph.parse(str(existing_db_abox_path), format=infer_rdf_format(existing_db_abox_path))
        detected_db_id = _detect_db_id_from_graph(operator)
        if detected_db_id:
            operator.db_id = detected_db_id
        operator.role_to_agent_id.update(_build_role_map_from_graph(operator))
    else:
        db_dir = benchmark_root / db_id
        operator.load_database_context(db_dir)
        if save_db_abox:
            generated_db_abox_path.parent.mkdir(parents=True, exist_ok=True)
            out_path = operator.save_db_abox(generated_db_abox_path, include_tbox=False)
            print(f"Saved DB ABOX: {out_path}")

    operator.prepare_reasoner()
    return operator


def extract_violations(status_map: dict[str, dict]) -> dict[str, dict]:
    violations = {}
    for key, details in (status_map or {}).items():
        status = str((details or {}).get("Status", "Aligned"))
        if status != "Aligned":
            violations[key] = details
    return violations

In [5]:
# ---- Path A: direct ArgosABoxOperator ----
operator = setup_argos_operator(
    ontology_path=ONTOLOGY_PATH,
    benchmark_root=BENCHMARK_ROOT,
    db_id=DB_ID,
    use_existing_db_abox=USE_EXISTING_DB_ABOX,
    existing_db_abox_path=EXISTING_DB_ABOX_PATH,
    save_db_abox=SAVE_DB_ABOX,
    generated_db_abox_path=GENERATED_DB_ABOX_PATH,
)

print(f"operator.db_id={operator.db_id}")
print(f"warmup_table_name={operator.warmup_table_name}")
print(f"known_roles={sorted(operator.role_to_agent_id.keys())[:10]}")

Saved DB ABOX: /home/nipdep/Dev/ARGOS/experiment/outputs/argos_notebook/financial_abox.rdf
table entities: {'order': 't007', 'loan': 't006', 'trans': 't008', 'client': 't003', 'disp': 't004', 'district': 't005', 'xxxxx': 't00x', 'account': 't001', 'card': 't002'}
column lookup: {('order', 'order_id'): 'c032', ('order', 'account_id'): 'c001', ('loan', 'account_id'): 'c001', ('trans', 'account_id'): 'c001', ('disp', 'account_id'): 'c001', ('account', 'account_id'): 'c001', ('order', 'bank_to'): 'c033', ('order', 'account_to'): 'c034', ('order', 'amount'): 'c028', ('loan', 'amount'): 'c028', ('trans', 'amount'): 'c028', ('order', 'k_symbol'): 'c035', ('trans', 'k_symbol'): 'c035', ('loan', 'duration'): 'c029', ('loan', 'loan_id'): 'c027', ('trans', 'balance'): 'c038', ('loan', 'date'): 'c004', ('trans', 'date'): 'c004', ('account', 'date'): 'c004', ('loan', 'payments'): 'c030', ('loan', 'status'): 'c031', ('client', 'birth_date'): 'c011', ('client', 'client_id'): 'c009', ('disp', 'client_

In [6]:
print("SELECT t1.\"amount\", t1.\"balance\", t1.\"date\", t1.\"type\"\nFROM \"trans\" AS t1\nWHERE t1.\"amount\" IS NOT NULL AND t1.\"trans_id\" IS NOT NULL AND t1.\"type\" IS NOT NULL")

SELECT t1."amount", t1."balance", t1."date", t1."type"
FROM "trans" AS t1
WHERE t1."amount" IS NOT NULL AND t1."trans_id" IS NOT NULL AND t1."type" IS NOT NULL


In [7]:
ROLE = "public"
SQL_QUERY = """
SELECT\n  amount,\n  balance,\n  date,\n  type\nFROM trans\nWHERE\n  NOT account IS NULL AND NOT amount IS NULL AND NOT trans_id IS NULL
""".strip()

# Evaluate one SQL query via ArgosABoxOperator
result = operator.evaluate_query(sql_query=SQL_QUERY, role=ROLE)
result_dict = result.to_dict()

table_violations = extract_violations(result_dict.get("table_status", {}))
column_violations = extract_violations(result_dict.get("column_status", {}))

print("\n=== ARGOS Operator Result ===")
print(f"role: {result_dict['role']}")
print(f"original_query: {result_dict['original_query']}")
print(f"refined_query: {result_dict['refined_query']}")
print(f"active_policies: {result_dict['active_policies']}")
print(f"active_rules: {result_dict['active_rules']}")
print(f"table_violation_count: {len(table_violations)}")
print(f"column_violation_count: {len(column_violations)}")

print("\n--- Table Violations ---")
pprint(table_violations)

print("\n--- Column Violations ---")
pprint(column_violations)

--> Instantiating ontology individuals...
SELECT amount, balance, date, type FROM trans WHERE NOT account IS NULL AND NOT amount IS NULL AND NOT trans_id IS NULL
Found 1 FROM/Join nodes in statement s981558.
Initial in-context sources: {'trans': {'type': 'base_table', 'name': 'trans'}}
Resolving column 'amount' in context of table alias 'None' | context_sources: {'trans': {'type': 'base_table', 'name': 'trans'}}
Resolving column 'balance' in context of table alias 'None' | context_sources: {'trans': {'type': 'base_table', 'name': 'trans'}}
Resolving column 'date' in context of table alias 'None' | context_sources: {'trans': {'type': 'base_table', 'name': 'trans'}}
Resolving column 'type' in context of table alias 'None' | context_sources: {'trans': {'type': 'base_table', 'name': 'trans'}}
Resolving column 'account' in context of table alias 'None' | context_sources: {'trans': {'type': 'base_table', 'name': 'trans'}}
Resolving column 'amount' in context of table alias 'None' | context_s

In [9]:
print(SQL_QUERY)

SELECT
  amount,
  balance,
  date,
  type
FROM trans
WHERE
  NOT account IS NULL AND NOT amount IS NULL AND NOT trans_id IS NULL


In [10]:
# ---- Path B: ArgosAccessController wrapper ----
controller = ArgosAccessController.from_benchmark_db(
    benchmark_root=str(BENCHMARK_ROOT),
    db_id=DB_ID,
    ontology_path=str(ONTOLOGY_PATH),
    save_db_abox=SAVE_DB_ABOX,
    db_abox_output_path=str(GENERATED_DB_ABOX_PATH) if SAVE_DB_ABOX else None,
)

decision = controller.refine_query(role=ROLE, sql_query=SQL_QUERY)
table_violations_ctrl = extract_violations(decision.get("table_status", {}))
column_violations_ctrl = extract_violations(decision.get("column_status", {}))

print("\n=== ArgosAccessController Decision ===")
print(f"status: {decision.get('status')}")
print(f"error: {decision.get('error')}")
print(f"refined_query: {decision.get('refined_query')}")
print(f"active_policies: {decision.get('active_policies')}")
print(f"active_rules: {decision.get('active_rules')}")
print(f"table_violation_count: {len(table_violations_ctrl)}")
print(f"column_violation_count: {len(column_violations_ctrl)}")

print("\n--- Table Violations ---")
pprint(table_violations_ctrl)
print("\n--- Column Violations ---")
pprint(column_violations_ctrl)

wrapper_output = run_argos_access_control_case(
    controller=controller,
    role=ROLE,
    sql_query=SQL_QUERY,
)
print("\n=== run_argos_access_control_case output ===")
print(json.dumps(wrapper_output, indent=2, ensure_ascii=False))

table entities: {'order': 't007', 'loan': 't006', 'trans': 't008', 'client': 't003', 'disp': 't004', 'district': 't005', 'xxxxx': 't00x', 'account': 't001', 'card': 't002'}
column lookup: {('order', 'order_id'): 'c032', ('order', 'account_id'): 'c001', ('loan', 'account_id'): 'c001', ('trans', 'account_id'): 'c001', ('disp', 'account_id'): 'c001', ('account', 'account_id'): 'c001', ('order', 'bank_to'): 'c033', ('order', 'account_to'): 'c034', ('order', 'amount'): 'c028', ('loan', 'amount'): 'c028', ('trans', 'amount'): 'c028', ('order', 'k_symbol'): 'c035', ('trans', 'k_symbol'): 'c035', ('loan', 'duration'): 'c029', ('loan', 'loan_id'): 'c027', ('trans', 'balance'): 'c038', ('loan', 'date'): 'c004', ('trans', 'date'): 'c004', ('account', 'date'): 'c004', ('loan', 'payments'): 'c030', ('loan', 'status'): 'c031', ('client', 'birth_date'): 'c011', ('client', 'client_id'): 'c009', ('disp', 'client_id'): 'c009', ('client', 'gender'): 'c010', ('client', 'district_id'): 'c002', ('district',

In [8]:
# Optional: quick batch sanity check
test_queries = [
    SQL_QUERY,
    "SELECT * FROM account;",
    "SELECT client_id, birth_number FROM client;",
]

batch_results = []
for idx, q in enumerate(test_queries, start=1):
    d = controller.refine_query(role=ROLE, sql_query=q)
    batch_results.append(
        {
            "idx": idx,
            "status": d.get("status"),
            "error": d.get("error"),
            "query": q,
            "refined_query": d.get("refined_query"),
            "active_policies": d.get("active_policies", []),
            "active_rules": d.get("active_rules", []),
        }
    )

print(json.dumps(batch_results, indent=2, ensure_ascii=False))

--> Instantiating ontology individuals...
SELECT l.account_id, o.account_to, o.k_symbol, MIN(o.amount) AS smallest_order_amount FROM loan AS l JOIN "order" AS o ON l.account_id = o.account_id GROUP BY l.account_id, o.account_to, o.k_symbol
Found 2 FROM/Join nodes in statement s03eeaf.
Initial in-context sources: {'loan': {'type': 'base_table', 'name': 'loan'}, 'l': {'type': 'base_table', 'name': 'loan'}, 'order': {'type': 'base_table', 'name': 'order'}, 'o': {'type': 'base_table', 'name': 'order'}}
Resolving column 'account_id' in context of table alias 'l' | context_sources: {'loan': {'type': 'base_table', 'name': 'loan'}, 'l': {'type': 'base_table', 'name': 'loan'}, 'order': {'type': 'base_table', 'name': 'order'}, 'o': {'type': 'base_table', 'name': 'order'}}
Resolving column 'account_to' in context of table alias 'o' | context_sources: {'loan': {'type': 'base_table', 'name': 'loan'}, 'l': {'type': 'base_table', 'name': 'loan'}, 'order': {'type': 'base_table', 'name': 'order'}, 'o':

In [9]:
# Cleanup (safe to run multiple times)
try:
    controller.close()
except Exception:
    pass

try:
    operator.close()
except Exception:
    pass

print("Closed controller/operator.")

Closed controller/operator.
